# 11.5 Embedding / Reranker 训练

> 🕐 预估学习时间：40分钟

RAG 质量上限很大程度上取决于嵌入与重排模型，而不仅是生成模型。本节用对比学习训练双塔检索器，并用 Cross-Encoder 做重排。

本节涵盖：
- 双塔 Dense Retriever
- InfoNCE / in-batch negatives
- Hard Negative Mining
- Cross-Encoder Reranker
- 与生成模型的联合评估


## 1. 双塔检索器 + InfoNCE

查询塔与文档塔共享或分离编码器，用 in-batch 负样本做对比学习。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class MeanPoolEncoder(nn.Module):
    def __init__(self, vocab=200, d=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.proj = nn.Linear(d, d)

    def forward(self, ids, mask):
        x = self.emb(ids)
        x = x * mask.unsqueeze(-1)
        pooled = x.sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        return F.normalize(self.proj(pooled), dim=-1)


def info_nce(q, d, temperature=0.05):
    logits = (q @ d.T) / temperature
    labels = torch.arange(q.size(0), device=q.device)
    return F.cross_entropy(logits, labels)


enc_q = MeanPoolEncoder()
enc_d = MeanPoolEncoder()
opt = torch.optim.Adam(list(enc_q.parameters()) + list(enc_d.parameters()), lr=1e-3)

print('=== Dual-Encoder InfoNCE ===')
for step in range(60):
    q_ids = torch.randint(0, 200, (32, 12))
    d_ids = torch.randint(0, 200, (32, 20))
    # plant a shared token to create weak positives
    shared = torch.randint(0, 200, (32, 1))
    q_ids[:, 0:1] = shared
    d_ids[:, 0:1] = shared
    q_mask = torch.ones_like(q_ids)
    d_mask = torch.ones_like(d_ids)
    q = enc_q(q_ids, q_mask)
    d = enc_d(d_ids, d_mask)
    loss = info_nce(q, d)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 15 == 0 or step == 59:
        with torch.no_grad():
            acc = (q @ d.T).argmax(-1).eq(torch.arange(32)).float().mean().item()
        print(f'step={step:02d} loss={loss.item():.4f} inbatch_acc={acc:.3f}')

print(f'\nKey: In-batch negatives make dual-encoder training scalable; temperature controls sharpness.')


## 2. Hard Negative Mining

随机负样本太易区分；用当前模型检索到的高分错误文档做难负样本，可显著提升分辨力。


In [ ]:
# Build a tiny corpus and mine hard negatives for a query batch
corpus_ids = torch.randint(0, 200, (200, 20))
corpus_mask = torch.ones_like(corpus_ids)
with torch.no_grad():
    corpus_vec = enc_d(corpus_ids, corpus_mask)

q_ids = torch.randint(0, 200, (8, 12))
q_ids[:, 0] = corpus_ids[:8, 0]  # positives = first 8 docs
q_mask = torch.ones_like(q_ids)
with torch.no_grad():
    q_vec = enc_q(q_ids, q_mask)
    sims = q_vec @ corpus_vec.T
    # top-3 including positive; pick highest non-positive as hard negative
    hard_negs = []
    for i in range(8):
        ranked = sims[i].argsort(descending=True).tolist()
        hard = next(j for j in ranked if j != i)
        hard_negs.append(hard)
print('=== Hard Negatives ===')
print(f'hard neg doc ids: {hard_negs}')
print(f'mean sim(q, pos)={[round((q_vec[i]@corpus_vec[i]).item(),3) for i in range(4)]}')
print(f'mean sim(q, hard)={[round((q_vec[i]@corpus_vec[hard_negs[i]]).item(),3) for i in range(4)]}')
print(f'\nKey: Hard negatives sit near the decision boundary and improve retrieval ranking quality.')


## 3. Cross-Encoder 重排

双塔快但未建模细交互；对 Top-K 候选用 Cross-Encoder 逐对打分重排。


In [ ]:
class CrossEncoder(nn.Module):
    def __init__(self, vocab=200, d=64):
        super().__init__()
        self.emb = nn.Embedding(vocab, d)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d, 4, 128, batch_first=True), 2
        )
        self.head = nn.Linear(d, 1)

    def forward(self, pair_ids, mask):
        x = self.emb(pair_ids)
        x = self.encoder(x, src_key_padding_mask=~mask.bool())
        pooled = (x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        return self.head(pooled).squeeze(-1)


ce = CrossEncoder()
opt_ce = torch.optim.Adam(ce.parameters(), lr=1e-3)
print('=== Cross-Encoder Reranker ===')
for step in range(40):
    # positive pairs vs random negatives
    pos_q = torch.randint(0, 200, (16, 8))
    pos_d = torch.randint(0, 200, (16, 12))
    neg_d = torch.randint(0, 200, (16, 12))
    pos_q[:, 0] = 1
    pos_d[:, 0] = 1  # shared marker
    pos_pair = torch.cat([pos_q, pos_d], dim=1)
    neg_pair = torch.cat([pos_q, neg_d], dim=1)
    mask = torch.ones(16, pos_pair.size(1))
    s_pos = ce(pos_pair, mask)
    s_neg = ce(neg_pair, mask)
    # pairwise logistic
    loss = -F.logsigmoid(s_pos - s_neg).mean()
    opt_ce.zero_grad()
    loss.backward()
    opt_ce.step()
    if step % 10 == 0 or step == 39:
        acc = (s_pos > s_neg).float().mean().item()
        print(f'step={step:02d} loss={loss.item():.4f} pairwise_acc={acc:.3f}')

print(f'\nKey: Retrieve wide with dual-encoder, then rerank narrow with cross-encoder.')


## 课后思考题

1. in-batch negative 在小 batch 或同质语料下有什么缺陷？如何补救？
2. Hard negative 过难（假阴性）时会怎样？如何过滤？
3. 何时该训练领域嵌入，而不是直接用通用 BGE/E5？
4. 如何用 nDCG/Recall@K 与下游 RAG 答案正确率做联合选模？

---
> 本节涵盖了11.5 Embedding / Reranker 训练的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
